# Estimate the ATM straddle using historical Realized Volatility

We're asked to make a market on 3 month ATM options on the SPX index.  
Our strategy will be to estimate recent realized volatility 
and use the Brenner and Subrahmanyan approximation:
$$
V = \frac{1}{\sqrt{2\pi}} S \sigma \sqrt{T}
$$
Since we only care about a very approximate answer we'll also take 
$\frac{1}{\sqrt{2\pi}}\approx0.4$ and $T=\frac{1}{4}$.

In [ ]:
# boilerplate
%load_ext autoreload
%autoreload 2
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import pandas as pd
import numpy as np

import bokeh
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
output_notebook(resources=bokeh.resources.INLINE, hide_banner=False, verbose=False)

Let's look at a year's worth of historical closing prices for the SPX index.  A simple csv download from Yahoo finance will suffice, symbol ^GSPC.

In [ ]:
df = pd.read_csv("SPX500.csv", parse_dates=['Date'])

In [ ]:
# Let's take a look at the data
df.head()

In [ ]:
# Looks like Close and Adjusted Close are the same, so we can just use the "Close".
# For equities with dividends and corporate actions, use Adjusted Close instead.
np.allclose(df['Close'].values, df['Adj Close'].values)

In [ ]:
# And plot the time series of the closing price 
fig1 = figure(title='SP500 Close', x_axis_label='Date', y_axis_label='Close', plot_width=800, plot_height=400, active_drag='box_zoom', x_axis_type='datetime')
fig1.line(df.Date, df.Close, legend_label='SP500', line_width=1)
show(fig1)

This exhibits the market catching a cold in mid-Feb, precipitously declining for about a month, then mostly recovering in a bumpy ride up, perhaps buoyed by Fed and Treasury stimulus.  We're interested in the volatility rather than market level per se. 

In [ ]:
# compute rolling 3 month std dev ... approx 21 trading days/month, 252 per year
N = 3 * 21
# Series of daily log-returns
log_returns = np.log(df['Close']).diff()[1:]
# rolling 3-month avg daily variance, x252 to annualize variance, sqrt to get volatility
realized_vol = np.sqrt(252 * log_returns.rolling(N).var())

In [ ]:
fig2 = figure(title='SP500 Rolling realized vol', x_axis_label='Date', y_axis_label='3M realized vol', plot_width=800, plot_height=400, active_drag='box_zoom', x_axis_type='datetime')
fig2.line(df.Date[N+1:], realized_vol[N:], legend_label='SP500', line_width=1)
show(fig2)

Compare to:
https://www.alphaquery.com/stock/SPY/volatility-option-statistics/90-day/historical-volatility

In [ ]:
realized_vol.describe()

This shows a remarkable range in realized volatility, from around 6% to 63%.  We'll use the most recent 3-month average we have, and take the most recent index value as the spot price:

In [ ]:
realized_vol.tail(1)

In [ ]:
df.tail(1)

Using Brenner and Subrahmanyan this gives us an ATM call or put price:

In [ ]:
# Call or put, use last index close value for S, last realized_vol for sigma
V = 0.4 * 3155.22 * 0.263255 * np.sqrt(1/4)  # call or put value
straddle = 2 * V  # call + put

In [ ]:
print(f"ATM call price: {V} straddle: {straddle}")

In [ ]:
#  Compare to current value of ATM implied vol
V_market = 0.4 * 3155.22 * 0.257346 * np.sqrt(0.25)

In [ ]:
print(f"ATM call market value (from implied vol): {V_market: .2f}")

In this case, the realized vol approach gets you in the ballpark (within a few percent.)  
In general it is not especially precise, but should give a feel for things.

Realized vol is backward looking, implied vol is forward looking.
So an anticipated event or period of stress can be reflected in implied vol even if not in recent realized vol. 